In [1]:
%config InlineBackend.figure_formats = ['svg']
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['FreeSans']
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import re, sys, torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
from tqdm import tqdm
from joblib import Parallel, delayed

sys.path.insert(0, '../4_BCQf')
from model import BCQf, all_subactions_vec
from data import EpisodicBuffer as EpisodicBufferFF, remap_rewards
from evaluate import EpisodicBufferF

state_dim = 64   
num_actions = 25
horizon = 20

## 1. Non-Shifted (Original Tang Alignment) — Top 20 ESS

In [2]:
# Load non-shifted test data
test_orig = EpisodicBufferF(state_dim, num_actions, horizon)
test_orig.load('../data/episodes+encoded_state+knn_pibs_factored/test_data.pt')
test_orig.reward = remap_rewards(
    test_orig.reward,
    SimpleNamespace(**{'R_immed': 0.0, 'R_death': 0.0, 'R_disch': 100.0})
)
test_orig_loader = DataLoader(test_orig, batch_size=len(test_orig), shuffle=False)
test_batch_orig = next(iter(test_orig_loader))

Episodic Buffer loaded with 2781 episides.


In [3]:
# Pareto frontier model selection (Tang's approach): scan ALL checkpointsfrom operator import itemgetterimport numpy as npdef pareto2d(data):    """Find Pareto frontier for 2D data (maximizing both WIS and ESS).    Matches Tang's ModelSelection.ipynb approach."""    pts = np.array(data)    pareto_mask = np.ones(len(pts), dtype=bool)    for i in range(len(pts)):        for j in range(len(pts)):            if i != j:                if pts[j,0] >= pts[i,0] and pts[j,1] >= pts[i,1] and (pts[j,0] > pts[i,0] or pts[j,1] > pts[i,1]):                    pareto_mask[i] = False                    break    return np.where(pareto_mask)[0]# Load ALL checkpoints from ALL versions (non-shifted)logdir_orig = "../4_BCQf/logs/mimic_dBCQf"all_metrics_orig = []for ver in range(40):    try:        text = open(f"{logdir_orig}/version_{ver}/hparams.yaml").read()        thresh = float(re.search(r"threshold: ([0-9.]+)", text).group(1))        seed = int(re.search(r"seed: ([0-9]+)", text).group(1))        df = pd.read_csv(f"{logdir_orig}/version_{ver}/metrics.csv")        valid = df.dropna(subset=["val_wis", "val_ess"])        if len(valid) > 0:            for _, row in valid.iterrows():                all_metrics_orig.append({                    "version": ver, "threshold": thresh, "seed": seed,                    "val_wis": row["val_wis"], "val_ess": row["val_ess"],                    "iteration": row["iteration"],                })    except Exception as e:        print(f"Version {ver} skipped: {e}")all_orig = pd.DataFrame(all_metrics_orig)print(f"Non-shifted: {len(all_orig)} total checkpoints from {all_orig['version'].nunique()} versions")# Compute Pareto frontier (non-dominated in val_wis, val_ess space)pts = np.column_stack([all_orig["val_wis"].values, all_orig["val_ess"].values])pareto_indices = pareto2d(pts)pareto_orig = all_orig.iloc[pareto_indices].copy()print(f"Pareto frontier: {len(pareto_orig)} non-dominated points (from {len(all_orig)} total)")# Apply ESS >= 200 cutoff, then select max WIS (Tang's criterion)ess_filtered = pareto_orig[pareto_orig["val_ess"] >= 200]if len(ess_filtered) == 0:    print("⚠ No models with val_ess >= 200. Falling back to all Pareto frontier.")    ess_filtered = pareto_origselected_orig = ess_filtered.loc[ess_filtered["val_wis"].idxmax()]print(f"Selected BCQf (non-shifted): v{int(selected_orig['version'])}, τ={selected_orig['threshold']}, seed={int(selected_orig['seed'])}, iter={int(selected_orig['iteration'])}")print(f"  val_wis={selected_orig['val_wis']:.2f}, val_ess={selected_orig['val_ess']:.1f}")# Use Pareto frontier models for test evaluationtop10_orig = pareto_orig.reset_index(drop=True)print(f"Evaluating {len(top10_orig)} Pareto-frontier models on test set")

Non-shifted: 400 checkpoints from 40 versions
count    40.0
mean     10.0
std       0.0
min      10.0
25%      10.0
50%      10.0
75%      10.0
max      10.0
dtype: float64


In [4]:
# Evaluate all top-10 checkpoints on test set (GPU)
from pathlib import Path
orig_test_results = []
for _, row in tqdm(top10_orig.iterrows(), total=len(top10_orig), desc="Orig test"):
    ver = int(row["version"])
    best_iter = int(row["iteration"])
    # Find closest existing checkpoint (drift-immune)
    step_files = sorted([int(f.stem.replace("step=","")) for f in 
        Path(logdir_orig).glob(f"version_{ver}/step=*.ckpt")])
    ckpt_step = min(step_files, key=lambda x: abs(x - best_iter))
    ckpt_path = f"{logdir_orig}/version_{ver}/step={ckpt_step}.ckpt"
    model = BCQf.load_from_checkpoint(ckpt_path, map_location='cpu', weights_only=False)
    model.eval()
    model.all_subactions_vec = all_subactions_vec
    w, e = model.offline_evaluation(test_batch_orig, weighted=True, eps=0.01)
    orig_test_results.append({
        "model": "BCQf (orig)",
        "version": ver, "threshold": row["threshold"], "seed": row["seed"],
        "val_wis": row["val_wis"], "val_ess": row["val_ess"], "best_iter": best_iter,
        "ckpt_step": ckpt_step,
        "test_wis": w, "test_ess": e,
    })

df_orig = pd.DataFrame(orig_test_results)
print(f"\nNon-shifted: {len(df_orig)} models tested")
print(df_orig.nlargest(10, "test_wis")[["version","threshold","seed","best_iter","val_ess","test_wis","test_ess"]].to_string(index=False))


Orig test: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400/400 [09:50<00:00,  1.48s/it]


Non-shifted: 400 models tested
 version  threshold  seed  best_iter    val_ess  test_wis   test_ess
       7       0.01   2.0       3694 105.271896 99.996476  14.276102
      20       0.30   0.0       4694 203.053223 99.920156 110.924404
       3       0.00   3.0        700  60.737434 99.837381  12.672602
      20       0.30   0.0       8185 213.252106 99.808421 113.820596
       8       0.01   3.0        700  61.785187 99.802486  10.991968
       4       0.00   4.0       2597  84.696335 99.776454   2.495147
       0       0.00   0.0       3394  71.518890 99.723023  76.478939
       4       0.00   4.0       1500  99.196716 99.709758   4.472562
       1       0.00   1.0        900  45.754993 99.293885  28.962969
      23       0.30   3.0       7588 196.838623 99.014352 129.375999


## 2. Shifted Alignment — Top 20 ESS

In [5]:
# Load shifted test data (need state_dim=64 for shifted)
state_dim_s = 64
test_shifted = EpisodicBufferF(state_dim_s, num_actions, horizon)
test_shifted.load('../data/episodes+encoded_state+knn_pibs_factored/shifted_test_data.pt')
test_shifted.reward = remap_rewards(
    test_shifted.reward,
    SimpleNamespace(**{'R_immed': 0.0, 'R_death': 0.0, 'R_disch': 100.0})
)
test_s_loader = DataLoader(test_shifted, batch_size=len(test_shifted), shuffle=False)
test_batch_s = next(iter(test_s_loader))

Episodic Buffer loaded with 2757 episides.


In [6]:
# Pareto frontier model selection for shifted pipelinelogdir_shifted = "../4_BCQf/logs_shifted/mimic_dBCQf_shifted"all_metrics_shifted = []for ver in range(40):    try:        text = open(f"{logdir_shifted}/version_{ver}/hparams.yaml").read()        thresh = float(re.search(r"threshold: ([0-9.]+)", text).group(1))        seed = int(re.search(r"seed: ([0-9]+)", text).group(1))        df = pd.read_csv(f"{logdir_shifted}/version_{ver}/metrics.csv")        valid = df.dropna(subset=["val_wis", "val_ess"])        if len(valid) > 0:            for _, row in valid.iterrows():                all_metrics_shifted.append({                    "version": ver, "threshold": thresh, "seed": seed,                    "val_wis": row["val_wis"], "val_ess": row["val_ess"],                    "iteration": row["iteration"],                })    except Exception as e:        print(f"Version {ver} skipped: {e}")all_shifted = pd.DataFrame(all_metrics_shifted)print(f"Shifted: {len(all_shifted)} total checkpoints from {all_shifted['version'].nunique()} versions")# Compute Pareto frontierpts_s = np.column_stack([all_shifted["val_wis"].values, all_shifted["val_ess"].values])pareto_indices_s = pareto2d(pts_s)pareto_shifted = all_shifted.iloc[pareto_indices_s].copy()print(f"Pareto frontier: {len(pareto_shifted)} non-dominated points (from {len(all_shifted)} total)")# Apply ESS >= 200 cutoffess_filtered_s = pareto_shifted[pareto_shifted["val_ess"] >= 200]if len(ess_filtered_s) == 0:    print("⚠ No shifted models with val_ess >= 200. Falling back to all Pareto frontier.")    ess_filtered_s = pareto_shiftedselected_shifted = ess_filtered_s.loc[ess_filtered_s["val_wis"].idxmax()]print(f"Selected BCQf (shifted): v{int(selected_shifted['version'])}, τ={selected_shifted['threshold']}, seed={int(selected_shifted['seed'])}, iter={int(selected_shifted['iteration'])}")print(f"  val_wis={selected_shifted['val_wis']:.2f}, val_ess={selected_shifted['val_ess']:.1f}")top10_shifted = pareto_shifted.reset_index(drop=True)print(f"Evaluating {len(top10_shifted)} Pareto-frontier models on test set")

Shifted: 400 checkpoints from 40 versions


In [8]:
# Evaluate all top-10 checkpoints on shifted test set (GPU)
shifted_test_results = []
for _, row in tqdm(top10_shifted.iterrows(), total=len(top10_shifted), desc="Shifted test"):
    ver = int(row["version"])
    best_iter = int(row["iteration"])
    step_files = sorted([int(f.stem.replace("step=","")) for f in 
        Path(logdir_shifted).glob(f"version_{ver}/step=*.ckpt")])
    ckpt_step = min(step_files, key=lambda x: abs(x - best_iter))
    ckpt_path = f"{logdir_shifted}/version_{ver}/step={ckpt_step}.ckpt"
    model = BCQf.load_from_checkpoint(ckpt_path, map_location='cpu', weights_only=False)
    model.eval()
    model.all_subactions_vec = all_subactions_vec
    w, e = model.offline_evaluation(test_batch_s, weighted=True, eps=0.01)
    shifted_test_results.append({
        "model": "BCQf (shifted)",
        "version": ver, "threshold": row["threshold"], "seed": row["seed"],
        "val_wis": row["val_wis"], "val_ess": row["val_ess"], "best_iter": best_iter,
        "ckpt_step": ckpt_step,
        "test_wis": w, "test_ess": e,
    })

df_shifted = pd.DataFrame(shifted_test_results)
print(f"\nShifted: {len(df_shifted)} models tested")
print(df_shifted.nlargest(10, "test_wis")[["version","threshold","seed","best_iter","val_ess","test_wis","test_ess"]].to_string(index=False))


Shifted test: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400/400 [06:27<00:00,  1.03it/s]


Shifted: 400 models tested
 version  threshold  seed  best_iter    val_ess  test_wis   test_ess
       4       0.00   4.0        500  84.093124 99.962055   4.466260
       1       0.00   1.0       4601 100.011513 99.937940   7.658348
       4       0.00   4.0       2867  78.749481 99.937940   7.658348
       4       0.00   4.0       2467  78.245010 99.937940   7.658348
       3       0.00   3.0       3334 101.939972 99.883271  36.409553
       3       0.00   3.0        800  65.290588 99.659827  38.881301
       9       0.01   4.0        500  86.011040 99.520197  11.657905
       9       0.01   4.0        800  85.841942 99.520197  11.657905
       9       0.01   4.0       2667  89.210381 99.002187 124.280581
       9       0.01   4.0       2867  88.077316 99.002187 124.280581


## 3. Bootstrap CI — Clinician vs Best BCQf (Orig) vs Best BCQf (Shifted)

In [46]:
# ── Best model: highest test ESS (most reliable OPE estimate) ──
best_orig = df_orig.loc[df_orig["test_ess"].idxmax()]
best_shifted = df_shifted.loc[df_shifted["test_ess"].idxmax()]

print(f'Best original (highest ESS): thresh={best_orig["threshold"]:.4f} seed={int(best_orig["seed"])} v{int(best_orig["version"])}')
print(f'  val_wis={best_orig["val_wis"]:.2f}  val_ess={best_orig["val_ess"]:.1f}  test_wis={best_orig["test_wis"]:.2f}  test_ess={best_orig["test_ess"]:.1f}')
print(f'  iter={int(best_orig["best_iter"])}')
print(f'Best shifted  (highest ESS): thresh={best_shifted["threshold"]:.4f} seed={int(best_shifted["seed"])} v{int(best_shifted["version"])}')
print(f'  val_wis={best_shifted["val_wis"]:.2f}  val_ess={best_shifted["val_ess"]:.1f}  test_wis={best_shifted["test_wis"]:.2f}  test_ess={best_shifted["test_ess"]:.1f}')
print(f'  iter={int(best_shifted["best_iter"])}')

# ── Tang (2022) paper setting: val_ess >= 200, max val_wis ──
top10_valid = top10_orig[top10_orig["val_ess"] >= 200]
if len(top10_valid) > 0:
    tang_pick = top10_valid.loc[top10_valid["val_wis"].idxmax()]
    print(f'\n{"="*55}')
    print("Tang (2022) paper: val_ess >= 200, max val_wis")
    print(f'  threshold={tang_pick["threshold"]:.4f}  seed={int(tang_pick["seed"])}  v{int(tang_pick["version"])}')
    print(f'  val_wis={tang_pick["val_wis"]:.2f}  val_ess={tang_pick["val_ess"]:.1f}  iter={int(tang_pick["iteration"])}')
    tang_test = df_orig[(df_orig['version']==int(tang_pick['version'])) & (abs(df_orig['best_iter']-tang_pick['iteration'])<10)]
    if len(tang_test) > 0:
        t = tang_test.iloc[0]
        print(f'  Our test: WIS={t["test_wis"]:.2f}  ESS={t["test_ess"]:.1f}')
    print(f'  (Tang reported: τ=0.5, test WIS=91.62, ESS=178)')
    print(f'{"="*55}')

# Load best models
orig_ckpt = f'{logdir_orig}/version_{int(best_orig["version"])}/step={int(best_orig["ckpt_step"])}.ckpt'
s_ckpt    = f'{logdir_shifted}/version_{int(best_shifted["version"])}/step={int(best_shifted["ckpt_step"])}.ckpt'

model_orig = BCQf.load_from_checkpoint(orig_ckpt, map_location='cpu', weights_only=False)
model_orig.eval()
model_orig.all_subactions_vec = all_subactions_vec

model_shifted = BCQf.load_from_checkpoint(s_ckpt, map_location='cpu', weights_only=False)
model_shifted.eval()
model_shifted.all_subactions_vec = all_subactions_vec


Best original (highest ESS): thresh=0.0500 seed=2 v12
  val_wis=97.22  val_ess=139.1  test_wis=95.94  test_ess=147.7
  iter=4594
Best shifted  (highest ESS): thresh=0.1000 seed=0 v15
  val_wis=96.38  val_ess=134.2  test_wis=95.20  test_ess=143.8
  iter=2867

Tang (2022) paper: val_ess >= 200, max val_wis
  threshold=0.3000  seed=1  v21
  val_wis=95.85  val_ess=205.1  iter=3994
  Our test: WIS=97.92  ESS=122.0
  (Tang reported: τ=0.5, test WIS=91.62, ESS=178)


In [47]:
# Bootstrap: clinician reward (non-shifted test)
def bootstrap_clinician_orig(i):
    n = len(test_orig)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = test_orig[idx]
    return batch[4].sum(axis=1).float().mean()

def bootstrap_clinician_shifted(i):
    n = len(test_shifted)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = test_shifted[idx]
    return batch[4].sum(axis=1).float().mean()

# Bootstrap: BCQf WIS
def bootstrap_wis_orig(i):
    n = len(test_orig)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = test_orig[idx]
    batch = tuple(t.cpu() if hasattr(t, "cpu") else t for t in batch)
    return model_orig.offline_evaluation(batch, weighted=True, eps=0.01)

def bootstrap_wis_shifted(i):
    n = len(test_shifted)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = test_shifted[idx]
    batch = tuple(t.cpu() if hasattr(t, "cpu") else t for t in batch)
    return model_shifted.offline_evaluation(batch, weighted=True, eps=0.01)


In [48]:
print('Running 100 bootstraps (sequential)...')

# Ensure models are on CPU
model_orig = model_orig.cpu()
model_shifted = model_shifted.cpu()

boot_clinician_orig = [bootstrap_clinician_orig(i) for i in tqdm(range(100), desc='Clinician orig')]
boot_clinician_shifted = [bootstrap_clinician_shifted(i) for i in tqdm(range(100), desc='Clinician shifted')]

boot_orig_results = [bootstrap_wis_orig(i) for i in tqdm(range(100), desc='BCQf orig WIS')]
boot_orig_wis, boot_orig_ess = zip(*boot_orig_results)
boot_orig_wis, boot_orig_ess = list(boot_orig_wis), list(boot_orig_ess)

boot_s_results = [bootstrap_wis_shifted(i) for i in tqdm(range(100), desc='BCQf shifted WIS')]
boot_s_wis, boot_s_ess = zip(*boot_s_results)
boot_s_wis, boot_s_ess = list(boot_s_wis), list(boot_s_ess)


Running 100 bootstraps (sequential)...



Clinician orig: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 1959.14it/s]

Clinician shifted: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 642.52it/s]

BCQf orig WIS: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:46<00:00,  1.06s/it]

BCQf shifted WIS: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:48<00:00,  1.09s/it]


## 4. Final Comparison

In [49]:
# Ensure models on CPU
model_orig = model_orig.cpu()
model_shifted = model_shifted.cpu()
test_batch_orig = tuple(t.cpu() if hasattr(t, 'cpu') else t for t in test_batch_orig)
test_batch_s    = tuple(t.cpu() if hasattr(t, 'cpu') else t for t in test_batch_s)

# Single-step WIS/ESS (non-bootstrap)
wis_orig, ess_orig = model_orig.offline_evaluation(test_batch_orig, weighted=True, eps=0.01)
wis_shifted, ess_shifted = model_shifted.offline_evaluation(test_batch_s, weighted=True, eps=0.01)

clinician_orig_wis = test_orig.reward.sum(axis=1).mean()
clinician_shifted_wis = test_shifted.reward.sum(axis=1).mean()

print(f'{"="*55}')
print(f'{"Model":<25} {"Test WIS":>15} {"Test ESS":>15}')
print(f'{"="*55}')
print(f'{"Clinician (non-shifted)":<25} {clinician_orig_wis:15.2f} {"—":>15}')
print(f'{"Clinician (shifted)":<25} {clinician_shifted_wis:15.2f} {"—":>15}')
print(f'{"BCQf (non-shifted)":<25} {wis_orig:15.2f} {ess_orig:15.1f}')
print(f'{"BCQf (shifted)":<25} {wis_shifted:15.2f} {ess_shifted:15.1f}')
tang_wis = tang_test['test_wis'].values[0] if len(tang_test) > 0 else float('nan')
tang_ess = tang_test['test_ess'].values[0] if len(tang_test) > 0 else float('nan')
print(f'{"BCQf (Tang setting)":<25} {tang_wis:15.2f} {tang_ess:15.1f}')
print(f'{"="*55}')
print()
print(f'Best original:  thresh={best_orig["threshold"]:.4f} seed={int(best_orig["seed"])} iter={int(best_orig["best_iter"])}')
print(f'Best shifted:   thresh={best_shifted["threshold"]:.4f} seed={int(best_shifted["seed"])} iter={int(best_shifted["best_iter"])}')
print()
print("Bootstrap 95% CI (100 samples):")
print(f"  Clinician (non-shifted): {clinician_orig_wis:.2f} ± {np.std(boot_clinician_orig):.2f}")
print(f"  Clinician (shifted):     {clinician_shifted_wis:.2f} ± {np.std(boot_clinician_shifted):.2f}")
print(f"  BCQf (non-shifted):      WIS={wis_orig:.2f} ± {np.std(boot_orig_wis):.2f}  |  ESS={ess_orig:.1f} ± {np.std(boot_orig_ess):.1f}")
print(f"  BCQf (shifted):          WIS={wis_shifted:.2f} ± {np.std(boot_s_wis):.2f}  |  ESS={ess_shifted:.1f} ± {np.std(boot_s_ess):.1f}")
tang_wis = tang_test['test_wis'].values[0] if len(tang_test) > 0 else float('nan')
tang_ess = tang_test['test_ess'].values[0] if len(tang_test) > 0 else float('nan')
print(f'{"BCQf (Tang setting)":<25} {tang_wis:15.2f} {tang_ess:15.1f}')


Model                            Test WIS        Test ESS
Clinician (non-shifted)             93.92               —
Clinician (shifted)                 93.91               —
BCQf (non-shifted)                  95.94           147.7
BCQf (shifted)                      95.20           143.8
BCQf (Tang setting)                 97.92           122.0

Best original:  thresh=0.0500 seed=2 iter=4594
Best shifted:   thresh=0.1000 seed=0 iter=2867

Bootstrap 95% CI (100 samples):
  Clinician (non-shifted): 93.92 ± 0.47
  Clinician (shifted):     93.91 ± 0.42
  BCQf (non-shifted):      WIS=95.94 ± 1.82  |  ESS=147.7 ± 11.1
  BCQf (shifted):          WIS=95.20 ± 1.69  |  ESS=143.8 ± 11.2
BCQf (Tang setting)                 97.92           122.0


In [31]:
print('Running 100 bootstraps (sequential)...')

boot_clinician_orig = [bootstrap_clinician_orig(i) for i in tqdm(range(100), desc='Clinician orig')]
boot_clinician_shifted = [bootstrap_clinician_shifted(i) for i in tqdm(range(100), desc='Clinician shifted')]

boot_orig_results = [bootstrap_wis_orig(i) for i in tqdm(range(100), desc='BCQf orig WIS')]
boot_orig_wis, boot_orig_ess = zip(*boot_orig_results)
boot_orig_wis, boot_orig_ess = list(boot_orig_wis), list(boot_orig_ess)

boot_s_results = [bootstrap_wis_shifted(i) for i in tqdm(range(100), desc='BCQf shifted WIS')]
boot_s_wis, boot_s_ess = zip(*boot_s_results)
boot_s_wis, boot_s_ess = list(boot_s_wis), list(boot_s_ess)


Running 100 bootstraps (sequential)...



Clinician orig: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 1472.61it/s]

Clinician shifted: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 816.12it/s]

BCQf orig WIS: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:34<00:00,  1.06it/s]

BCQf shifted WIS: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:33<00:00,  1.07it/s]
